# PathSnap: Multi-Model Zoo & Experimental Comparison (Table 3.1)
**Course:** UE24CS352A - Machine Learning | PES University  
**Author:** A S Vincent (`vincent2maran@gmail.com`)

This notebook implements:
1. Feature extraction across multiple training trips.
2. Group-Aware Cross-Validation (`GroupKFold` strictly by `trip_id` to prevent data leakage).
3. Implementation and evaluation of **5 DISTINCT MODELS across multiple algorithm families**:
   - **Model 1:** Random Forest Classifier (Tree-based Ensemble)
   - **Model 2:** LightGBM Classifier (Gradient Boosting)
   - **Model 3:** Multi-Layer Perceptron (Neural Network)
   - **Model 4:** Distance-Weighted KNN (Instance-based)
   - **Model 5:** Logistic Regression (Linear & Probabilistic Baseline)
4. Full Table 3.1 Model Comparison Matrix (Macro F1, Top-1 Accuracy, Log Loss, Latency).


In [1]:
# Setup dependencies
!pip install -q pyproj networkx scikit-learn lightgbm pandas numpy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from pathlib import Path

LOCAL_DATA = Path("../Student folder/Student folder/train")
if not LOCAL_DATA.exists():
    LOCAL_DATA = Path("data/train")


'pip' is not recognized as an internal or external command,
operable program or batch file.


## 1. Feature Extraction Pipeline Definition


In [2]:
from scipy.spatial import cKDTree
from pyproj import Transformer, CRS

def point_to_segment_projection(px, py, x1, y1, x2, y2):
    dx, dy = x2 - x1, y2 - y1
    l2 = dx*dx + dy*dy
    if l2 == 0:
        return float(np.hypot(px - x1, py - y1)), 0.0
    u = ((px - x1)*dx + (py - y1)*dy) / l2
    u_clamped = max(0.0, min(1.0, u))
    proj_x = x1 + u_clamped * dx
    proj_y = y1 + u_clamped * dy
    dist = float(np.hypot(px - proj_x, py - proj_y))
    return dist, float(u_clamped)

def extract_features_for_trip(trip_dir, trip_id, candidate_radius=60.0, top_k=10):
    nodes = np.loadtxt(trip_dir / f"{trip_id}.nodes")
    arcs = np.loadtxt(trip_dir / f"{trip_id}.arcs", dtype=np.int64)
    track = np.loadtxt(trip_dir / f"{trip_id}.track")
    route = np.loadtxt(trip_dir / f"{trip_id}.route", dtype=np.int64)

    # UTM projection
    zone = int((np.mean(nodes[:, 0]) + 180) / 6) + 1
    tf = Transformer.from_crs(CRS.from_epsg(4326), CRS.from_epsg(32600 + zone), always_xy=False)
    nodes_x, nodes_y = tf.transform(nodes[:, 1], nodes[:, 0])
    track_x, track_y = tf.transform(track[:, 1], track[:, 0])

    # Sample KD-Tree
    pts, seg_ids = [], []
    for arc_id in range(len(arcs)):
        fn, tn = arcs[arc_id]
        x0, y0 = nodes_x[fn], nodes_y[fn]
        x1, y1 = nodes_x[tn], nodes_y[tn]
        L = np.hypot(x1 - x0, y1 - y0)
        n_s = max(2, int(np.ceil(L / 20.0)) + 1)
        for a in np.linspace(0, 1, n_s):
            pts.append((x0 + a*(x1 - x0), y0 + a*(y1 - y0)))
            seg_ids.append(arc_id)
    tree = cKDTree(np.array(pts))
    seg_ids = np.array(seg_ids)

    # Monotonic route alignment
    n_obs = len(track)
    n_route = len(route)
    aligned_arcs = np.empty(n_obs, dtype=np.int64)
    r_idx = 0
    for i in range(n_obs):
        px, py = track_x[i], track_y[i]
        best_d = float('inf')
        best_arc = route[r_idx]
        best_r = r_idx
        for cand_r in range(max(0, r_idx - 2), min(n_route, r_idx + 16)):
            arc_id = route[cand_r]
            fn, tn = arcs[arc_id]
            d, _ = point_to_segment_projection(px, py, nodes_x[fn], nodes_y[fn], nodes_x[tn], nodes_y[tn])
            if d < best_d:
                best_d = d
                best_arc = arc_id
                best_r = cand_r
        aligned_arcs[i] = best_arc
        r_idx = best_r

    dx = np.diff(track_x, append=track_x[-1])
    dy = np.diff(track_y, append=track_y[-1])
    headings = np.arctan2(dy, dx)

    rows = []
    for i in range(n_obs):
        px, py = track_x[i], track_y[i]
        true_arc = aligned_arcs[i]
        nearby = tree.query_ball_point([px, py], r=candidate_radius)
        cand_arcs = list(set(seg_ids[nearby])) if nearby else [seg_ids[tree.query([px, py], k=1)[1]]]

        scored = []
        for arc_id in cand_arcs:
            fn, tn = arcs[arc_id]
            d, u = point_to_segment_projection(px, py, nodes_x[fn], nodes_y[fn], nodes_x[tn], nodes_y[tn])
            scored.append((arc_id, d, u, np.hypot(nodes_x[tn]-nodes_x[fn], nodes_y[tn]-nodes_y[fn]), np.arctan2(nodes_y[tn]-nodes_y[fn], nodes_x[tn]-nodes_x[fn])))
        scored.sort(key=lambda x: x[1])
        scored = scored[:top_k]

        for rank, (arc_id, d, u, L, road_head) in enumerate(scored):
            head_diff = abs((headings[i] - road_head + np.pi) % (2.0 * np.pi) - np.pi)
            rows.append({
                "trip_id": trip_id,
                "observation_idx": i,
                "arc_id": arc_id,
                "dist_perp_m": d,
                "proj_ratio_u": u,
                "road_length_m": L,
                "heading_diff_deg": np.degrees(head_diff),
                "cos_heading_diff": np.cos(head_diff),
                "candidate_rank": rank,
                "is_true_match": int(arc_id == true_arc)
            })
    return pd.DataFrame(rows)


## 2. Extract Training Features Across Trips


In [3]:
# Extract dataset for 5 trips (adjust trip count as needed)
trip_list = sorted([d.name for d in LOCAL_DATA.iterdir() if d.is_dir()])[:5]
print("Building dataset from trips:", trip_list)

dfs = []
for t_id in trip_list:
    df_t = extract_features_for_trip(LOCAL_DATA / t_id, t_id)
    dfs.append(df_t)

full_df = pd.concat(dfs, ignore_index=True)
print(f"Total candidate rows: {len(full_df):,}")
print("Class balance (0: Negative, 1: True Match):")
print(full_df['is_true_match'].value_counts())


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'data\\train'

## 3. Train & Evaluate the 5 Distinct Models with GroupKFold


In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, log_loss, roc_auc_score, precision_score, recall_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier

FEATURE_COLS = ["dist_perp_m", "proj_ratio_u", "road_length_m", "heading_diff_deg", "cos_heading_diff", "candidate_rank"]

models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=12, class_weight='balanced_subsample', random_state=42, n_jobs=-1),
    "LightGBM": LGBMClassifier(n_estimators=120, max_depth=6, learning_rate=0.08, random_state=42, n_jobs=-1, verbose=-1),
    "Multi-Layer Perceptron": Pipeline([("scaler", StandardScaler()), ("mlp", MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=150, random_state=42))]),
    "Distance-Weighted KNN": Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=9, weights='distance', n_jobs=-1))]),
    "Logistic Regression": Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(class_weight='balanced', max_iter=300, random_state=42))]),
}

comparison_rows = []
X = full_df[FEATURE_COLS].values
y = full_df["is_true_match"].values
groups = full_df["trip_id"].values
gkf = GroupKFold(n_splits=5)

for name, model in models.items():
    print(f"Running 5-Fold GroupKFold CV for: {name}...")
    f1s, top1s, losses, times = [], [], [], []

    for tr_idx, va_idx in gkf.split(X, y, groups):
        model.fit(X[tr_idx], y[tr_idx])

        t0 = time.perf_counter()
        if hasattr(model, "predict_proba"):
            probs = model.predict_proba(X[va_idx])[:, 1]
        else:
            probs = model.predict(X[va_idx]).astype(float)
        latency = (time.perf_counter() - t0)

        preds = (probs >= 0.5).astype(int)
        f1s.append(f1_score(y[va_idx], preds, average="macro", zero_division=0))
        losses.append(log_loss(y[va_idx], np.clip(probs, 1e-7, 1 - 1e-7)))

        # Top-1 accuracy
        val_df = full_df.iloc[va_idx].copy()
        val_df["prob"] = probs
        top_picks = val_df.loc[val_df.groupby(["trip_id", "observation_idx"])["prob"].idxmax()]
        top1s.append(top_picks["is_true_match"].mean())
        times.append(latency / val_df.groupby(["trip_id", "observation_idx"]).ngroups * 1000.0)

    comparison_rows.append({
        "Model Name": name,
        "CV Macro F1": round(np.mean(f1s), 4),
        "Top-1 Accuracy (%)": round(np.mean(top1s) * 100, 2),
        "Log Loss": round(np.mean(losses), 4),
        "Inference Latency (ms/obs)": round(np.mean(times), 2)
    })

comparison_df = pd.DataFrame(comparison_rows)
print("\n================ TABLE 3.1: MODEL COMPARISON MATRIX ================")
print(comparison_df.to_string(index=False))
comparison_df.to_csv("table_3_1_model_comparison.csv", index=False)


## 4. Visualization: Comparative Performance Plots


In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.barh(comparison_df["Model Name"], comparison_df["CV Macro F1"], color='steelblue')
plt.title("Cross-Validation Macro F1 Score")
plt.xlabel("Macro F1")

plt.subplot(1, 2, 2)
plt.barh(comparison_df["Model Name"], comparison_df["Top-1 Accuracy (%)"], color='coral')
plt.title("Observation-level Top-1 Accuracy (%)")
plt.xlabel("Top-1 Accuracy %")
plt.tight_layout()
plt.show()
